⏱️ **Time required:** ~10 minutes | **Type:** Interactive tutorial

# Compliance & Governance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/02_compliance_governance.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/02_compliance_governance.ipynb)

GDPR erasure in 2 lines, automatic lineage, zero-retention architectures, automated PII handling, and per-entity cost intelligence.

In [1]:
import subprocess
import sys
import importlib
import urllib.request
import os

if importlib.util.find_spec("lakelogic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "-q", "lakelogic[polars]"])
if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

lakelogic v1.20.0 | Local | c:\_Personal\_SaaS\lakelogic\examples\colab


---
## 1. GDPR Erasure -- `forget_subjects()` in 2 Lines

**Scenario:** A customer emails your DPO: *"Delete all my data."* Under GDPR Article 17, you have 30 days. Your data lives across 14 tables in 3 layers. How fast can you comply?

LakeLogic scans every PII-flagged field in a contract, applies the chosen erasure strategy (`nullify`, `hash`, or `redact`), and writes an immutable audit log -- all in two lines of code.


In [2]:
from lakelogic.core.gdpr import forget_subjects

contract_path = s.write_contract(
    """
version: 1.0.0
dataset: customers

model:
  fields:
    - name: customer_id
      type: string
      required: true
    - name: name
      type: string
      pii: true
    - name: email
      type: string
      pii: true
    - name: phone
      type: string
      pii: true
    - name: lifetime_value
      type: float
""",
    "02_compliance_governance_demo/customers.yaml",
)

# Generate a dataset with PII
source_df = ll.DataGenerator(contract_path).generate(rows=100)
proc = ll.DataProcessor(contract_path, engine="polars")
good, _ = proc.run(source_df)

import polars as pl

sample_id = good["customer_id"][0]
print(f"Before erasure ({sample_id}):")

display(
    good.filter(pl.col("customer_id") == sample_id).select(["customer_id", "name", "email", "phone", "lifetime_value"])
)

2026-04-27 13:28:26.695 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: customers
2026-04-27 13:28:26.695 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 100 valid + 0 invalid = 100 total
2026-04-27 13:28:26.695 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-27 13:28:26.724 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 100 records built
2026-04-27 13:28:26.756 | WARNING  | lakelogic.core.masking_engine:apply:377 - PII fields detected without masking strategy: [name, email, phone]. Set 'masking:' (nullify|hash|redact|partial|encrypt) in your contract to enable masking for these fields.
2026-04-27 13:28:26.757 | INFO     | lakelogic.core.masking_engine:apply:384 - No PII fields with explicit masking strategy — skipping masking.
2026-04-27 13:28:26.757 | INFO     | lakelogic.core.processor:run:818 - Run complete | Source: 

Before erasure (CUST-357298):


customer_id,name,email,phone,lifetime_value
str,str,str,str,f64
"""CUST-357298""","""Diana Larson""","""william04@example.org""","""7204381826""",427.4099


In [ ]:
# The Proof — erasure + audit trail
erased = forget_subjects(
    good,
    proc.contract,
    subject_column="customer_id",
    subject_ids=[sample_id],
    erasure_strategy="hash",
)

print(f"After erasure (subject: {sample_id}):")
audit_cols = [
    c
    for c in erased.columns
    if "customer_id" in c or "name" in c or "email" in c or "phone" in c or "lifetime_value" in c or "_lakelogic_" in c
]
display(erased.filter(pl.col("customer_id") == sample_id).select(audit_cols[:8]))

print("\nPII fields hashed. Audit columns added. Erasure in 2 lines of code.")

---
## 2. Zero-Retention Architecture -- Source Purge After Ingestion

**But wait** -- the GDPR erasure above cleaned the *tables*. What about the raw source files still sitting in the landing zone? That's unmasked PII in blob storage, months after ingestion.

With `server.post_ingestion.action: delete`, LakeLogic purges source files **immediately after a successful Bronze commit**. The cleanup is transactional -- if the commit fails, files are preserved.

| Action | Effect | Use Case |
| :--- | :--- | :--- |
| `delete` | Remove source files after commit | Zero-retention / PII compliance |
| `archive` | Move source files to archive path | Audit trail with cold storage |
| `retain` | Leave source files in place (default) | Development, debugging |


In [ ]:
from pathlib import Path

# Use the demo directory as the landing zone
# In Colab: visible in /content/lakelogic_demo/...
# Locally:  visible in examples/colab/02_compliance_governance_demo/landing/
landing = Path("02_compliance_governance_demo/landing")
landing.mkdir(parents=True, exist_ok=True)
landing_str = str(landing.resolve()).replace("\\", "/")

# Contract: post_ingestion lives directly on the source block
# No server block needed for simple pipelines!
zr_contract = s.write_contract(
    f"""
version: 1.0.0
dataset: transient_events

source:
  type: local
  format: parquet
  path: "{landing_str}"
  post_ingestion:
      action: delete                # delete | archive | retain
      cleanup_is_blocking: false    # cleanup failure ≠ pipeline failure
      # archive_path: "{landing_str}/archive"

model:
  fields:
    - name: event_id
      type: string
      required: true
    - name: event_type
      type: string
    - name: payload
      type: string
""",
    "02_compliance_governance_demo/transient_events.yaml",
)

# Step 1: Generate test data into the landing zone
source_df = ll.DataGenerator(zr_contract).generate(rows=30)
source_df.write_parquet(str(landing / "batch_001.parquet"))

files_before = list(landing.glob("*.parquet"))
print(f"Landing zone BEFORE ingestion: {len(files_before)} file(s)")
print(f"  Location: {landing.resolve()}")
for f in files_before:
    print(f"   {f.name} ({f.stat().st_size:,} bytes)")

# Step 2: Run the pipeline -- reads from source.path automatically
proc = ll.DataProcessor(zr_contract, engine="polars")
good, bad = proc.run_source()

print(f"\nPipeline result: {len(good)} rows ingested, {len(bad)} quarantined")
display(good.select(["event_id", "event_type", "payload"]).head(5))

# In production, PipelineRunner executes post_ingestion after commit:
#   source.post_ingestion  -- simple contract-level setup
#   server.post_ingestion  -- system-level override (data mesh)
print("\nIn production, post_ingestion.action=delete purges these files")
print("automatically after a successful Bronze Delta commit.")
print("\nChange action to 'retain' above and re-run to keep the files.")

---
## 3. Automated PII Handling -- Masking at Ingestion Time

**So far** we can erase PII on demand (Section 1) and purge raw source files (Section 2). But what about the data that *does* get stored in Bronze/Silver/Gold tables? If a downstream analyst queries it, they see raw PII.

Declare `pii: true` and a `masking` strategy per-field in the contract. LakeLogic applies masking **automatically during every `run()`** -- the same contract works on Polars, Spark, and DuckDB without code changes.

| Strategy | Effect | Use Case |
| :--- | :--- | :--- |
| `hash` | SHA-256 (irreversible, deterministic) | Join keys that must remain linkable across tables |
| `partial` | `j***@company.com` (custom format template) | Support teams who need to identify without seeing full PII |
| `redact` | Replaced with `***REDACTED***` | Fields with zero business need downstream |
| `nullify` | Set to `NULL` | Sensitive numeric fields (salary, SSN) |
| `encrypt` | AES-256 Fernet (reversible via key) | Re-identification by authorized compliance officers |


In [ ]:
import os
import polars as pl

# In production, inject keys via Azure Key Vault / AWS Secrets Manager / Databricks secret scope
os.environ["LAKELOGIC_PII_KEY"] = "demo-key-32-bytes-long-1234567!"
os.environ["LAKELOGIC_PII_SALT"] = "demo-salt"

# ── Contract: each PII field declares its own masking strategy ────
pii_contract = s.write_contract(
    """
version: 1.0.0
dataset: employees_pii

model:
  fields:
    - name: employee_id
      type: string
      required: true
    - name: full_name
      type: string
      pii: true
      masking: "hash"              # SHA-256 -- deterministic, allows cross-table joins
    - name: email
      type: string
      pii: true
      masking: "partial"           # j***@company.com -- identifiable for support
      masking_format: "{first1}***@{domain}"
    - name: phone
      type: string
      pii: true
      masking: "redact"            # Replaced with ***REDACTED***
    - name: salary
      type: float
      pii: true
      masking: "nullify"           # Set to NULL -- no downstream access
    - name: department
      type: string
""",
    "02_compliance_governance_demo/employees_pii.yaml",
)

# Generate synthetic employees and run through the pipeline
source_df = ll.DataGenerator(pii_contract).generate(rows=20)
proc = ll.DataProcessor(pii_contract, engine="polars")
good, bad = proc.run(source_df)

# ── Before: raw PII visible in the source data ────────────────────
print("BEFORE masking (raw source data):")
display(source_df.select(["employee_id", "full_name", "email", "phone", "salary", "department"]).head(5))

# ── After: PII masked automatically by the pipeline ─────────────
print("\nAFTER masking (what gets written to the table):")
display(good.select(["employee_id", "full_name", "email", "phone", "salary", "department"]).head(5))

# ── Summary ──────────────────────────────────────────────────
strategies = proc.last_report.get("trace_steps", [])
masking_step = next((s for s in strategies if s.get("step") == "PII Masking"), {})
print(f"\nMasking applied: {masking_step.get('details', {}).get('strategies', {})}")
print("\nIn production, this runs identically on Spark, Polars, or DuckDB -- same contract, zero code changes.")
print("Configure keys via: LAKELOGIC_PII_KEY env var, Azure Key Vault, or Databricks secret scope.")

---
## 4. Automatic Lineage -- Trace Any Row to Its Source

**Now prove it.** An auditor asks: *"This row in the gold table -- where did it come from, when was it processed, and who triggered the pipeline?"* With Sections 1-3 handling the data, you need a provenance trail to prove compliance.

Enable `lineage.enabled: true` in the contract. LakeLogic stamps every output row with source path, processing timestamp, run ID, and contract name -- automatically, on every run.


In [ ]:
# Run a pipeline and inspect lineage columns
lineage_contract = s.write_contract(
    """
version: 1.0.0
dataset: orders_lineage
info:
  title: silver_orders
  target_layer: silver

model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
    - name: status
      type: string

quality:
  row_rules:
    - name: positive
      sql: "amount > 0"

lineage:
  enabled: true
  upstream: [bronze.raw_orders]

""",
    "02_compliance_governance_demo/lineage.yaml",
)

source_df = ll.DataGenerator(lineage_contract).generate(rows=50)
proc = ll.DataProcessor(lineage_contract, engine="polars")
good, bad = proc.run(source_df, source_path="bronze/raw_orders/")

In [ ]:
# The Proof — lineage columns on every row
lineage_cols = [c for c in good.columns if "_lakelogic_" in c]
print(f"Lineage columns added automatically: {lineage_cols}")
print()
display(good.select(["order_id"] + lineage_cols).head(5))
print("\nEvery row traceable to its source. Every run has a unique ID.")

---
## 5. Pipeline Cost Intelligence

**The final piece:** all this compliance machinery -- erasure, purging, masking, lineage -- runs on cloud compute. When the CFO asks *"What does GDPR compliance actually cost us or domain (e.g marketing) pipelines cost for last month/last week/ yesterday ?"*, you need per-entity cost attribution.

LakeLogic tracks estimated cost per pipeline run using DBU rates, cluster configuration, and actual duration. No external billing APIs required -- just declare `cost.rates` in the contract metadata.


In [ ]:
# Run two contracts with cost tracking enabled
small = s.write_contract(
    """
version: 1.0.0
dataset: small_entity
info:
  title: bronze_small_entity
  domain: marketing
  system: google_analytics

metadata:
  domain: marketing
  system: google_analytics
  data_layer: bronze
  cost:
    provider: "manual"
    currency: "USD"
    rates:
      dbu_per_hour: 0.22

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: value
      type: string
""",
    "02_compliance_governance_demo/small.yaml",
)

large = s.write_contract(
    """
version: 1.0.0
dataset: large_entity
info:
  title: bronze_large_entity
  domain: finance
  system: shopify

metadata:
  domain: finance
  system: shopify
  data_layer: bronze
  cost:
    provider: "manual"
    currency: "USD"
    rates:
      dbu_per_hour: 0.55
    cluster:
      min_nodes: 2
      max_nodes: 8
      scaling_assumption: "avg"

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: value
      type: string
""",
    "02_compliance_governance_demo/large.yaml",
)

p1 = ll.DataProcessor(small, engine="polars")
p1.run(ll.DataGenerator(small).generate(rows=100))
r1 = p1.last_report

p2 = ll.DataProcessor(large, engine="polars")
p2.run(ll.DataGenerator(large).generate(rows=5000))
r2 = p2.last_report

# ── Per-Entity Cost Report ──────────────────────────────────────
print("Per-Entity Cost Attribution")
print("=" * 60)
for label, r in [("marketing / small_entity", r1), ("finance   / large_entity", r2)]:
    counts = r.get("counts", {})
    cost = r.get("estimated_cost", 0) or 0
    currency = r.get("cost_currency", "USD") or "USD"
    confidence = r.get("cost_confidence", "none")
    duration = r.get("run_duration_seconds", 0)
    print(f"  {label}")
    print(f"    Rows     : {counts.get('source', '?'):>6}")
    print(f"    Duration : {duration:.3f}s")
    print(f"    Cost     : {currency} {cost:.6f}  (confidence: {confidence})")
    print()

print("In production, these cost estimates flow into the run log")
print("and feed domain-level budget dashboards.")
print("\nConfigure cost.provider in _system.yaml:")
print("  manual       → duration × DBU rate × nodes")
print("  databricks   → queries system.billing.usage for exact costs")

## What You Just Saw

- **GDPR erasure** — hash/nullify PII with an audit trail, in 2 lines
- **Automatic lineage** — run ID and timestamp on every row, no config needed
- **Zero-Retention Architecture** — built-in `zero_retention_days` enforcement for transient data layers
- **Automated PII Handling** — declarative encryption and hashing applied at the Bronze layer
- **Cost intelligence** — per-entity, per-domain attribution in every run report

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.